# 132STA - Direct stiffness method for a trusses

Apply the matrix approach to the direct stiffness method on a problem shown during the lecture.

### 0. Preparation

Run the compact **Setup** cell once, then run the imports below. Expand the setup code only if you want to inspect it. In Colab, it loads the teaching functions at commit `cf36c7d`; locally, it uses your checkout.

In [ ]:
# @title Setup: load teaching functions (run first) { display-mode: "form" }
# Locate the teaching package (Colab or local Jupyter)
from pathlib import Path
import subprocess
import sys

revision = "cf36c7d744b5f80b14384b3a6bd9df684f98e81d"  # Original Lecture 05 truss functions.
repo = Path.cwd()
if not (repo / "sta132").is_dir():
    if (repo.parent / "sta132").is_dir():
        repo = repo.parent
    else:
        repo = Path(f"/content/132sta-python-05-{revision[:7]}")
        if not (repo / ".git").is_dir():
            subprocess.run(
                ["git", "clone", "--quiet", "--no-checkout",
                 "https://github.com/MartinDoskar/132sta-python.git", str(repo)],
                check=True,
            )
            subprocess.run(
                ["git", "checkout", "--quiet", "--detach", revision],
                cwd=repo, check=True,
            )
        actual_revision = subprocess.check_output(
            ["git", "rev-parse", "HEAD"], cwd=repo, text=True,
        ).strip()
        if actual_revision != revision:
            raise RuntimeError("The cached teaching package is not at the Lecture 05 commit.")
sys.path.insert(0, str(repo))
print("Using teaching functions from:", repo)


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sta132.truss import stiffness_matrix_truss, axial_force_truss
from sta132.plotting import plot_truss
np.set_printoptions(precision=4, suppress=True)

### 1. Define the problem/structure

The structure is defined with coordinates of the joints and elements defined by their endpoints. 
Plus, define the material and cross-sectional characteristics.

In [ ]:
# Node coordinates [x, z] in metres
xz = np.array([[0., 0.],   # Node 1
               [0., 3.],   # Node 2
               [4., 3.]])  # Node 3

# Member endpoints, using zero-based Python node indices
bars = np.array([[0, 2],   # Member 1--3
                 [1, 2],   # Member 2--3
                 [1, 0]])  # Member 2--1

E = 200e6  # kN/m^2
A = 3e-3   # m^2


### 2. Member stiffness relations

Compute stiffness matrices of each truss element in the global coordinate system.

In [ ]:
K1, Kloc1, T1 = stiffness_matrix_truss(E, A, xz[0], xz[2])
K2, Kloc2, T2 = stiffness_matrix_truss(E, A, xz[1], xz[2])
K3, Kloc3, T3 = stiffness_matrix_truss(E, A, xz[1], xz[0])

print("K1:\n", K1)
print("K2:\n", K2)
print("K3:\n", K3)

### 3. Compatibility, equilibrium, and code numbers

Construct the global stiffness matrix using manually provided code numbers.
Keep the one-based code numbers separate from the zero-based Python indices.

In [ ]:
K = np.zeros((6, 6))

code_numbers1 = np.array([1, 2, 5, 6])  # Member 1--3
code_numbers2 = np.array([3, 4, 5, 6])  # Member 2--3
code_numbers3 = np.array([3, 4, 1, 2])  # Member 2--1

# Lecture code numbers start at 1; Python array indices start at 0.
indices1 = code_numbers1 - 1
indices2 = code_numbers2 - 1
indices3 = code_numbers3 - 1

K[np.ix_(indices1, indices1)] += K1
K[np.ix_(indices2, indices2)] += K2
K[np.ix_(indices3, indices3)] += K3

print("K [10^3 kN/m]:\n", K / 1e3)

### 4. Supports and applied load

Solve the system by extracting a subsystem pertinent to free DOFs.

In [ ]:
p = np.zeros(6)    # Applied nodal loads [kN]
p[4] = 270.        # Horizontal load at node 3 (code number 5)

d = np.zeros(6)    # Nodal displacements [m]

free_code_numbers = np.array([2, 5])
fixed_code_numbers = np.array([1, 3, 4, 6])

# Lecture code numbers start at 1; Python array indices start at 0.
free_indices = free_code_numbers - 1
fixed_indices = fixed_code_numbers - 1

Kuu = K[np.ix_(free_indices, free_indices)]
print("Reduced K [10^3 kN/m]:\n", Kuu / 1e3)
print("Reduced load vector [kN]:", p[free_indices])

d[free_indices] = np.linalg.solve(Kuu, p[free_indices])
print("All nodal displacements [mm]:", 1e3 * d)
print(f"w1 = {1e3*d[1]:.4f} mm, u3 = {1e3*d[4]:.4f} mm")

### 5. Support reactions

Obtain reactions by back-substitution into the global (unpartitioned) system.

In [ ]:
r = K @ d - p
print("Reaction vector [kN]:", r)
print("Support reactions [kN]:", r[fixed_indices])
print("Residual at free DOFs [kN]:", r[free_indices])

### 6. Member axial forces

Use solution to compute internal forces.

In [ ]:
N1 = axial_force_truss(E, A, xz[0], xz[2], d[indices1])
N2 = axial_force_truss(E, A, xz[1], xz[2], d[indices2])
N3 = axial_force_truss(E, A, xz[1], xz[0], d[indices3])
N = np.array([N1, N2, N3])
print("Member forces [kN]:", N)

### 7. Plot the solution

Need scaling of the displacements

In [ ]:
fig, ax = plot_truss(xz, bars, d, scale=500)
plt.show()